# Custom Chatbot Project

TODO: In this cell, write an explanation of which dataset you have chosen and why it is appropriate for this task.



In [ ]:
```
https://www.dndbeyond.com/srd
https://media.wizards.com/2023/downloads/dnd/SRD_CC_v5.1.pdf
```
This work includes material taken from the System Reference Document 5.1 (“SRD 5.1”) by Wizards of
the Coast LLC and available at https://dnd.wizards.com/resources/systems-reference-document. The
SRD 5.1 is licensed under the Creative Commons Attribution 4.0 International License available at
https://creativecommons.org/licenses/by/4.0/legalcode.

In [111]:
import os
import json
import zipfile
import requests
import openai
import pandas as pd
import tiktoken

from pprint import pprint
from scipy import spatial

In [2]:
openai.api_base = "https://openai.vocareum.com/v1"
openai.api_key = "YOUR API KEY"

## Data Wrangling

TODO: In the cells below, load your chosen dataset into a `pandas` dataframe with a column named `"text"`. This column should contain all of your text data, separated into at least 20 rows.

In [3]:
data_file_path = "../data/dnd_srd_wiki.zip"
if os.path.exists(data_file_path):
    print(f"using existing data: {data_file_path=}")
else:
    print(f"downloading data: {data_file_path=}")
    url = "https://github.com/oldmanumby/DND.SRD.Wiki/archive/refs/heads/main.zip"
    with open(data_file_path, mode="wb") as file:
        res = requests.get(url)
        # res.content is the response body as bytes
        file.write(res.content)

using existing data: data_file_path='../data/dnd_srd_wiki.zip'


In [69]:
exceptions = []
text_pages = []
with zipfile.ZipFile(data_file_path, "r") as zip_file:
    infos = zip_file.infolist()
    for index, info in enumerate(infos):
        # only read markdown files and skip the changelog
        if (
            info.filename.endswith(".md") and 
            "/Spells/" in info.filename and
            "/Spells/#" not in info.filename
        ):
            # print(info.filename)
            try:
                byte_data = zip_file.read(info.filename)
                text = byte_data.decode("utf-8")
                # print("\n--------------------")
                # print(text)
                text_pages.append(text)
                # this breaks a page into smaller sections for embeddings,
                # but is not needed for spells.
                # text_pages += map(
                #     lambda s: s if s.startswith("#") else "## " + s,
                #     text.strip().split("## ")
                # )
            except Exception as ex:
                exceptions.append(ex)

print(f"\n{len(exceptions)=}")

df1 = pd.DataFrame(text_pages, columns=["text"])
df1 = df1[df1["text"].str.len() > 0].reset_index(drop=True)

df1.to_csv("../data/pages.csv")

df1


len(exceptions)=0


,text
0,### Acid Arrow\n\n*2nd-level evocation*\n\n**C...
1,### Acid Splash\n\n*Conjuration cantrip*\n\n**...
2,### Aid\n\n*2nd-level abjuration*\n\n**Casting...
3,### Alarm\n\n*1st-level abjuration (ritual)*\n...
4,### Alter Self\n\n*2nd-level transmutation*\n\...
...,...
314,### Wind Walk\n\n*6th-level transmutation*\n\n...
315,### Wind Wall\n\n*3rd-level evocation*\n\n**Ca...
316,### Wish\n\n*9th-level conjuration*\n\n**Casti...
317,### Word of Recall\n\n*6th-level conjuration*\...


## Custom Query Completion

TODO: In the cells below, compose a custom query using your chosen dataset and retrieve results from an OpenAI `Completion` model. You may copy and paste any useful code from the course materials.

In [146]:
question_1 = "What is the fire spell causes the most damage?"

question_2 = "What is the spell with the greatest range?"


basic_prompt_template = """
Question: {}
Answer: 
"""

custom_prompt_template = """
Answer the question based on the context below, and if the question
can't be answered based on the context, say "I don't know"

Context:
{}

---
Question: {}
Answer:
"""

In [127]:
if not os.path.exists("../data/embeddings.csv"):
    embeddings = []
    for text in df1["text"].tolist():
        embeddings_res = openai.Embedding.create(
            engine="text-embedding-ada-002",
            input=[text],
            encoding_format="float",
        )
        # simple progress bar...
        print(".", end="")
        # print(embeddings_res)
        if "data" in embeddings_res:
            embeddings.append(embeddings_res["data"][0]["embedding"])
        else:
            print(embeddings_res)
            print(text)
            break
    
    df2 = df1.copy()
    df2["embeddings"] = embeddings
    df2.to_csv("../data/embeddings.csv")

In [128]:
df2 = pd.read_csv("../data/embeddings.csv", converters={"embeddings": lambda x: json.loads(x)})
df2

,Unnamed: 0,text,embeddings
0,0,### Acid Arrow\n\n*2nd-level evocation*\n\n**C...,"[0.037330292, -0.0057263486, 0.019470256, -0.0..."
1,1,### Acid Splash\n\n*Conjuration cantrip*\n\n**...,"[0.028600873, 0.018988675, 0.033289112, -0.034..."
2,2,### Aid\n\n*2nd-level abjuration*\n\n**Casting...,"[0.007957265, 0.00067648577, 0.0038670818, 0.0..."
3,3,### Alarm\n\n*1st-level abjuration (ritual)*\n...,"[0.009247305, -0.0020823188, 0.019834798, -0.0..."
4,4,### Alter Self\n\n*2nd-level transmutation*\n\...,"[0.028250787, 0.021747775, 0.025825484, -0.002..."
...,...,...,...
314,314,### Wind Walk\n\n*6th-level transmutation*\n\n...,"[0.019588768, -0.004047405, 0.0030363936, 0.01..."
315,315,### Wind Wall\n\n*3rd-level evocation*\n\n**Ca...,"[0.008738997, 0.0030716774, -0.0011157588, 0.0..."
316,316,### Wish\n\n*9th-level conjuration*\n\n**Casti...,"[0.017392024, 0.008138231, 0.033198066, -0.003..."
317,317,### Word of Recall\n\n*6th-level conjuration*\...,"[0.033542663, -0.01743739, 0.010650261, -0.007..."


In [129]:
def distances_from_embeddings(
    query_embedding: list[float],
    embeddings: list[list[float]],
    distance_metric="cosine",
) -> list[float]:
    distance_metrics = {
        "cosine": spatial.distance.cosine,
        "L1": spatial.distance.cityblock,
        "L2": spatial.distance.euclidean,
        "Linf": spatial.distance.chebyshev,
    }
    distances = [
        distance_metrics[distance_metric](query_embedding, embedding)
        for embedding in embeddings
    ]
    return distances


In [130]:
embeddings_res = openai.Embedding.create(
    engine="text-embedding-ada-002",
    input=[question_1],
    encoding_format="float",
)
# print(embeddings_res)
question_embedding = embeddings_res["data"][0]["embedding"]
distances = distances_from_embeddings(
    question_embedding,
    df2["embeddings"].tolist(),
)

df3 = df2.copy()
df3["distances"] = distances
df3.sort_values(by="distances", ascending=True, inplace=True)

df3.to_csv("../data/custom_question_1_distances.csv")
df3

,Unnamed: 0,text,embeddings,distances
115,115,### Fire Bolt\n\n*Evocation cantrip*\n\n**Cast...,"[0.008499305, 0.0070696375, -0.011358638, -0.0...",0.173037
118,118,### Fireball\n\n*3rd-level evocation*\n\n**Cas...,"[0.020884693, 0.0001495735, -0.003088558, -0.0...",0.180070
117,117,### Fire Storm\n\n*7th-level evocation*\n\n**C...,"[0.013768468, -0.0018537028, -0.014060286, -0....",0.181417
230,230,### Produce Flame\n\n*Conjuration cantrip*\n\n...,"[0.00873013, -0.0067144777, -0.0013154094, -0....",0.184791
304,304,### Wall of Fire\n\n*4th-level evocation*\n\n*...,"[0.020694701, 0.007518813, -0.012999512, -0.01...",0.185939
...,...,...,...,...
8,8,### Animate Dead\n\n*3rd-level necromancy*\n\n...,"[-0.0006007953, -0.016741263, 0.0095924735, -0...",0.282257
253,253,### Secret Chest\n\n*4th-level conjuration*\n\...,"[0.013811594, 0.0003340561, 0.028947586, -0.02...",0.282896
134,134,### Gentle Repose\n\n*2nd-level necromancy (ri...,"[0.0091901235, -0.0014650178, 0.041820534, -0....",0.285334
265,265,### Silent Image\n\n*1st-level illusion*\n\n**...,"[0.0079655815, 0.005069609, 0.009662078, -0.00...",0.286981


In [164]:
tokenizer = tiktoken.get_encoding("cl100k_base")
max_token_count = 4000 - 200 # 200 for the completion

In [165]:

current_token_count = len(tokenizer.encode(prompt_template.format("", question_1)))

text_lines = []
for text in df3["text"]:
    count = len(tokenizer.encode("\n---\n" + text))
    if current_token_count + count < max_token_count:
        text_lines.append("\n---\n" + text)
        current_token_count += count
    else:
        break

custom_prompt_1 = custom_prompt_template.format("".join(text_lines), question_1)
print(custom_prompt_1[0:1024])


Answer the question based on the context below, and if the question
can't be answered based on the context, say "I don't know"

Context:

---
### Fire Bolt

*Evocation cantrip*

**Casting Time:** 1 action

**Range:** 120 feet

**Components:** V, S

**Duration:** Instantaneous

You hurl a mote of fire at a creature or object within range. Make a ranged spell attack against the target. On a hit, the target takes 1d10 fire damage. A flammable object hit by this spell ignites if it isn't being worn or carried. 

This spell's damage increases by 1d10 when you reach 5th level (2d10), 11th level (3d10), and 17th level (4d10).
---
### Fireball

*3rd-level evocation*

**Casting Time:** 1 action

**Range:** 150 feet

**Components:** V, S, M (a tiny ball of bat guano and sulfur)

**Duration:** Instantaneous

A bright streak flashes from your pointing finger to a point you choose within range and then blossoms with a low roar into an explosion of flame. Each creature in a 20-foot radius sphere ce

In [166]:
embeddings_res = openai.Embedding.create(
    engine="text-embedding-ada-002",
    input=[question_2],
    encoding_format="float",
)
# print(embeddings_res)
question_embedding = embeddings_res["data"][0]["embedding"]
distances = distances_from_embeddings(
    question_embedding,
    df2["embeddings"].tolist(),
)

df4 = df2.copy()
df4["distances"] = distances
df4.sort_values(by="distances", ascending=True, inplace=True)

df4.to_csv("../data/custom_question_2_distances.csv")
df4

,Unnamed: 0,text,embeddings,distances
189,189,### Magic Missile\n\n*1st-level evocation*\n\n...,"[-0.0011524442, -0.00039816904, -0.0104022175,...",0.175981
125,125,### Fog Cloud\n\n*1st-level conjuration*\n\n**...,"[0.03560496, 0.01261172, 0.00537987, -0.009612...",0.179013
115,115,### Fire Bolt\n\n*Evocation cantrip*\n\n**Cast...,"[0.008499305, 0.0070696375, -0.011358638, -0.0...",0.180244
138,138,### Glyph of Warding\n\n*3rd-level abjuration*...,"[0.022904705, 0.011499788, 0.0063835005, -0.01...",0.182140
239,239,### Ray of Frost\n\n*Evocation cantrip*\n\n**C...,"[0.0035988267, -0.02009616, -0.004223578, -0.0...",0.182443
...,...,...,...,...
270,270,### Spare the Dying\n\n*Necromancy cantrip*\n\...,"[0.0072914287, -0.00059705426, 0.034373876, -0...",0.239983
247,247,### Revivify\n\n*3rd-level necromancy*\n\n**Ca...,"[0.013563574, 0.00017061694, 0.021825513, 0.01...",0.240127
134,134,### Gentle Repose\n\n*2nd-level necromancy (ri...,"[0.0091901235, -0.0014650178, 0.041820534, -0....",0.243762
8,8,### Animate Dead\n\n*3rd-level necromancy*\n\n...,"[-0.0006007953, -0.016741263, 0.0095924735, -0...",0.246135


In [167]:
current_token_count = len(tokenizer.encode(prompt_template.format("", question_2)))

text_lines = []
for text in df4["text"]:
    count = len(tokenizer.encode("\n---\n" + text))
    if current_token_count + count < max_token_count:
        text_lines.append("\n---\n" + text)
        current_token_count += count
    else:
        break

custom_prompt_2 = custom_prompt_template.format("".join(text_lines), question_2)
print(custom_prompt_2[0:1024])


Answer the question based on the context below, and if the question
can't be answered based on the context, say "I don't know"

Context:

---
### Magic Missile

*1st-level evocation*

**Casting Time:** 1 action

**Range:** 120 feet

**Components:** V, S

**Duration:** Instantaneous

You create three glowing darts of magical force. Each dart hits a creature of your choice that you can see within range. A dart deals 1d4 + 1 force damage to its target. The darts all strike simultaneously, and you can direct them to hit one creature or several.

***At Higher Levels***. When you cast this spell using a spell slot of 2nd level or higher, the spell creates one more dart for each slot level above 1st.
---
### Fog Cloud

*1st-level conjuration*

**Casting Time:** 1 action

**Range:** 120 feet

**Components:** V, S

**Duration:** Concentration, up to 1 hour

You create a 20-foot radius sphere of fog centered on a point within range. The sphere spreads around corners, and its area is heavily obs

## Custom Performance Demonstration

TODO: In the cells below, demonstrate the performance of your custom query using at least 2 questions. For each question, show the answer from a basic `Completion` model query as well as the answer from your custom query.

### Question 1

In [168]:
openai.Completion.create(model="gpt-3.5-turbo-instruct")
basic_answer_1 = openai.Completion.create(
    model="gpt-3.5-turbo-instruct",
    prompt=basic_prompt_1,
    max_tokens=200,
)
# pprint(basic_answer_1)
basic_answer_1_text = basic_answer_1["choices"][0]["text"]
print("basic_answer_1...\n", basic_answer_1_text)

basic_answer_1...
 The fire spell that causes the most damage is up for debate, as it depends on various factors such as casting ability, target's resistance to fire, and level of the spell. However, some commonly known powerful fire spells include Inferno, Meteor, and Firestorm.


In [174]:
openai.Completion.create(model="gpt-3.5-turbo-instruct")
custom_answer_1 = openai.Completion.create(
    model="gpt-3.5-turbo-instruct",
    prompt=custom_prompt_1,
    max_tokens=200,
)
# pprint(custom_answer_1)
custom_answer_1_text = custom_answer_1["choices"][0]["text"]
print("custom_answer_1...\n", custom_answer_1_text)

custom_answer_1...
 
Meteor Swarm


### Question 2

In [171]:
openai.Completion.create(model="gpt-3.5-turbo-instruct")
basic_answer_2 = openai.Completion.create(
    model="gpt-3.5-turbo-instruct",
    prompt=basic_prompt_template.format(question_2),
    max_tokens=200,
)
# pprint(basic_answer_2)
basic_answer_2_text = basic_answer_2["choices"][0]["text"]
print("basic_answer_2...\n", basic_answer_2_text)

basic_answer_2...
 The spell with the greatest range is the "Astral Projection" spell, which allows the caster to project their consciousness to any location in the universe, regardless of distance, and observe or interact with their surroundings. This spell has an infinite range and is considered one of the most powerful and advanced spells in magic.


In [173]:
openai.Completion.create(model="gpt-3.5-turbo-instruct")
custom_answer_2 = openai.Completion.create(
    model="gpt-3.5-turbo-instruct",
    prompt=custom_prompt_2,
    max_tokens=200,
)
# pprint(custom_answer_2)
custom_answer_2_text = custom_answer_2["choices"][0]["text"]
print("custom_answer_2...\n", custom_answer_2_text)

custom_answer_2...
 Glyph of Warding and Find the Path both have a range of Self and allow the user to determine the location of a specific fixed location on the same plane of existence, which means potentially infinite distance. However, Fire Bolt has a range of 120 feet, which is the highest numerical range mentioned in the context. Therefore, Fire Bolt has the greatest range.
